[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C15_Classic_Architectures_Course/01_convolution_pooling/01_convolution_pooling.ipynb)

# 01 · 卷积与池化（纯 numpy 从零）

目标：把**互相关 vs 卷积**、**padding/stride/输出尺寸**、**多通道与参数量**、**感受野**、**max/avg 池化**、以及**前向 + 反向**全部用 numpy 从零实现，并与 `scipy.signal` 对拍、用**数值梯度检验**验证反向。

路线：互相关 from scratch（对拍 scipy）→ 卷积=翻转核的互相关 → padding/stride/输出尺寸 → 多通道+参数量 → 池化前向 → conv 反向（数值检验）→ 池化反向 → ✏️ 练习 → 📖 答案 → 🧪 optdigits 真实数据胶囊。

> 心智模型：**卷积 = 一小块核滑遍全图、逐位相乘求和**；框架的「卷积」其实是**不翻转核的互相关**。

## 0 · 公共工具：数值梯度检验 + 对拍裁判

先备好两件全模块通用的工具（与模块 00 一致）：中心差分**数值梯度**（验证反向的金标准）与 **对拍**裁判。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def numerical_grad(f, x, eps=1e-5):
    '''中心差分逐元素估计 df/dx。f: ndarray->标量。会原地探测后复原 x。'''
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps; fp = f(x)
        x[idx] = old - eps; fm = f(x)
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

def rel_error(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.max(np.abs(a - b) / np.maximum(1e-8, np.abs(a) + np.abs(b)))

def check_allclose(name, got, ref, atol=1e-10):
    got, ref = np.asarray(got, float), np.asarray(ref, float)
    ok = np.allclose(got, ref, atol=atol)
    print(f'[{name:<30}] allclose={ok}  max|err|={np.max(np.abs(got-ref)) if got.size else 0:.2e}')
    assert ok, f'{name} 与参考不一致!'
    return ok
print('✅ 工具就绪')

## 1 · 互相关 from scratch（对拍 scipy）

互相关：核**不翻转**地滑过输入，逐位相乘求和。输出位置 `(i,j)`：

$$ (X \star K)[i,j] = \sum_{a,b} X[i+a, j+b]\,K[a,b] $$

我们写一个朴素双重循环版，对拍 `scipy.signal.correlate2d(..., mode='valid')`。

In [ ]:
def corr2d(X, K):
    '''单通道 valid 互相关（不翻转核）。X:(H,W) K:(kh,kw) -> (H-kh+1, W-kw+1)。'''
    H, W = X.shape; kh, kw = K.shape
    oh, ow = H - kh + 1, W - kw + 1
    Y = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            Y[i, j] = np.sum(X[i:i+kh, j:j+kw] * K)   # 逐位乘求和
    return Y

X = rng.standard_normal((6, 7))
K = rng.standard_normal((3, 3))
Y = corr2d(X, K)
print('输出形状', Y.shape, '= (6-3+1, 7-3+1) =', (6-3+1, 7-3+1))

from scipy.signal import correlate2d
Y_scipy = correlate2d(X, K, mode='valid')
check_allclose('corr2d vs scipy.correlate2d', Y, Y_scipy)
print('✅ 我们的互相关 == scipy 的 correlate2d（这正是框架 Conv2d 真正做的事）')

## 2 · 数学卷积 = 翻转核的互相关

数学上的**卷积**要先把核翻转 180° 再做互相关。我们验证：`corr2d(X, flip(K)) == scipy.signal.convolve2d(X,K,'valid')`。

并验证一个性质差异：**卷积满足交换律**（`X*K == K*X`），**互相关不满足**。

In [ ]:
from scipy.signal import convolve2d

def flip2d(K):
    return K[::-1, ::-1]            # 上下 + 左右翻转

def conv2d_true(X, K):
    '''真·数学卷积 = 先翻转核再互相关。'''
    return corr2d(X, flip2d(K))

Y_conv = conv2d_true(X, K)
Y_conv_scipy = convolve2d(X, K, mode='valid')
check_allclose('conv(翻转核) vs scipy.convolve2d', Y_conv, Y_conv_scipy)

# 卷积交换律成立、互相关不成立（用同尺寸小核演示 full 模式更直观）
a = rng.standard_normal((5, 5)); b = rng.standard_normal((3, 3))
comm_conv = np.allclose(convolve2d(a, b, 'full'), convolve2d(b, a, 'full'))
comm_corr = np.allclose(correlate2d(a, b, 'full'), correlate2d(b, a, 'full'))
print('卷积交换律 X*K==K*X ?', comm_conv)
print('互相关交换律      ?', comm_corr)
assert comm_conv and not comm_corr
print('✅ 卷积=翻转核的互相关；卷积可交换、互相关不可交换。框架用互相关（更直观，核反正是学的）')

## 3 · padding / stride 与输出尺寸公式

输出尺寸（一维）：`out = floor((in + 2p - k)/s) + 1`。

实现带 padding 和 stride 的互相关，并验证输出尺寸；`same` padding（`p=(k-1)//2, s=1`）应保持尺寸不变。

In [ ]:
def out_size(in_size, k, s=1, p=0):
    return (in_size + 2 * p - k) // s + 1

def corr2d_ps(X, K, stride=1, pad=0):
    '''带 padding + stride 的 valid 互相关。'''
    if pad > 0:
        X = np.pad(X, pad, mode='constant')
    H, W = X.shape; kh, kw = K.shape
    oh, ow = out_size(H, kh, stride), out_size(W, kw, stride)   # 注意 X 已含 pad
    Y = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            r, cc = i * stride, j * stride
            Y[i, j] = np.sum(X[r:r+kh, cc:cc+kw] * K)
    return Y

X = rng.standard_normal((8, 8)); K = rng.standard_normal((3, 3))
# valid: 8->6
assert corr2d_ps(X, K, 1, 0).shape == (6, 6)
# same (p=1,s=1): 8->8
assert corr2d_ps(X, K, 1, 1).shape == (8, 8)
# stride 2, pad1: floor((8+2-3)/2)+1 = 4
assert corr2d_ps(X, K, 2, 1).shape == (4, 4)
print('valid 8->', corr2d_ps(X,K,1,0).shape[0], '| same 8->', corr2d_ps(X,K,1,1).shape[0], '| s=2,p=1 8->', corr2d_ps(X,K,2,1).shape[0])
# 不整除时地板取整丢边：in=8,k=3,s=3 -> floor(5/3)+1 = 2
assert out_size(8, 3, 3, 0) == 2
print('✅ 输出尺寸公式 + padding/stride 全部验证（含不整除截断）')

## 4 · 多通道卷积与参数量

一层有 `C_out` 个核，每核形状 `(C_in, kh, kw)`。输出通道 = 各输入通道互相关后**沿通道求和** + 偏置。

参数量 `= C_out*C_in*kh*kw + C_out`，**与 H、W 无关**（参数共享的威力）。

In [ ]:
def conv2d_multi(X, W, b, stride=1, pad=0):
    '''X:(C_in,H,Wd)  W:(C_out,C_in,kh,kw)  b:(C_out,) -> Y:(C_out,oh,ow)'''
    C_in, H, Wd = X.shape
    C_out, C_in2, kh, kw = W.shape
    assert C_in == C_in2, '核的通道深度必须等于输入通道数'
    Y = None
    for o in range(C_out):
        acc = None
        for ci in range(C_in):
            part = corr2d_ps(X[ci], W[o, ci], stride, pad)
            acc = part if acc is None else acc + part      # 沿通道求和
        acc = acc + b[o]
        if Y is None:
            Y = np.zeros((C_out,) + acc.shape)
        Y[o] = acc
    return Y

def conv_params(C_out, C_in, kh, kw):
    return C_out * C_in * kh * kw + C_out

X = rng.standard_normal((3, 8, 8))           # RGB-like 3 通道 8x8
W = rng.standard_normal((5, 3, 3, 3))        # 5 个核, 每核 (3,3,3)
b = rng.standard_normal(5)
Y = conv2d_multi(X, W, b, stride=1, pad=1)   # same
print('输入', X.shape, '-> 输出', Y.shape, '(5 通道, 8x8 保持)')
assert Y.shape == (5, 8, 8)
params = conv_params(5, 3, 3, 3)
print('该层参数量 =', params, '= 5*3*3*3 + 5 =', 5*3*3*3 + 5)
assert params == 140
# 关键：换更大的图，参数量不变
assert conv_params(5, 3, 3, 3) == 140      # 与 H,W 无关
print('✅ 多通道卷积正确；参数量只随 (C_out,C_in,kh,kw)，与图像尺寸无关')

## 5 · 池化前向：max 与 avg

在 `(k×k)` 窗口内取最大或取平均，步长通常 `s=k`（不重叠）。无可学习参数。

实现两种池化前向，并验证输出尺寸与具体数值。

In [ ]:
def pool2d(X, k=2, stride=2, mode='max'):
    '''单通道池化。X:(H,W) -> (oh,ow)。'''
    H, W = X.shape
    oh, ow = (H - k) // stride + 1, (W - k) // stride + 1
    Y = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            win = X[i*stride:i*stride+k, j*stride:j*stride+k]
            Y[i, j] = win.max() if mode == 'max' else win.mean()
    return Y

X = np.array([[1., 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12],
              [13, 14, 15, 16]])
Ymax = pool2d(X, 2, 2, 'max')
Yavg = pool2d(X, 2, 2, 'avg')
print('max pool:\n', Ymax)
print('avg pool:\n', Yavg)
assert Ymax.shape == (2, 2)
assert np.allclose(Ymax, [[6, 8], [14, 16]])          # 每 2x2 块的最大
assert np.allclose(Yavg, [[3.5, 5.5], [11.5, 13.5]])  # 每 2x2 块的均值
print('✅ max/avg 池化前向正确')

## 6 · 卷积反向：从零推导 + 数值梯度检验

给定 `dY`（损失对输出的梯度），推导：
- **对核**：`dK = corr2d(X, dY)`（输入与 dY 的互相关）
- **对输入**：`dX = full_conv(dY, K)`（dY 与**翻转核**的 full 卷积）

用标量损失 `L = sum(Y)`（则 `dY=1`）和**数值梯度**逐一验证。

In [ ]:
def corr2d_backward(X, K, dY):
    '''valid 互相关 Y=corr2d(X,K) 的反向。返回 dX, dK。'''
    H, W = X.shape; kh, kw = K.shape; oh, ow = dY.shape
    dX = np.zeros_like(X)
    dK = np.zeros_like(K)
    for i in range(oh):
        for j in range(ow):
            dK += dY[i, j] * X[i:i+kh, j:j+kw]          # 每个输出把窗口加权进 dK
            dX[i:i+kh, j:j+kw] += dY[i, j] * K          # 每个输出把梯度撒回窗口(权重=核)
    return dX, dK

X = rng.standard_normal((6, 6)); K = rng.standard_normal((3, 3))
Y = corr2d(X, K)
dY = np.ones_like(Y)                  # L = sum(Y) => dL/dY = 1
dX, dK = corr2d_backward(X, K, dY)

# 数值检验：dX 与 dK 各自对拍有限差分
L_of_X = lambda Xv: np.sum(corr2d(Xv, K))
L_of_K = lambda Kv: np.sum(corr2d(X, Kv))
dX_num = numerical_grad(L_of_X, X.copy())
dK_num = numerical_grad(L_of_K, K.copy())
print('dX 相对误差', f'{rel_error(dX, dX_num):.2e}')
print('dK 相对误差', f'{rel_error(dK, dK_num):.2e}')
assert rel_error(dX, dX_num) < 1e-6 and rel_error(dK, dK_num) < 1e-6

# 验证 dK 确实 = corr2d(X, dY)；dX 确实 = full conv(dY, flip K)
check_allclose('dK == corr2d(X,dY)', dK, correlate2d(X, dY, 'valid'))
check_allclose('dX == full conv(dY, K)', dX, convolve2d(dY, K, 'full'))
print('✅ 卷积反向 dX/dK 同时通过数值检验与解析公式对拍')

## 7 · 池化反向：梯度路由（无乘法）

池化没有参数，反向只是**把梯度分发**到输入：
- **max**：梯度只回流到窗口内 **argmax** 位置，其余 0
- **avg**：梯度**均分**给窗口内每个位置（除以 k*k）

用数值梯度验证两者。

In [ ]:
def pool2d_backward(X, dY, k=2, stride=2, mode='max'):
    dX = np.zeros_like(X)
    oh, ow = dY.shape
    for i in range(oh):
        for j in range(ow):
            r, cc = i*stride, j*stride
            win = X[r:r+k, cc:cc+k]
            if mode == 'max':
                am = np.unravel_index(np.argmax(win), win.shape)  # 只给最大位置
                dX[r+am[0], cc+am[1]] += dY[i, j]
            else:
                dX[r:r+k, cc:cc+k] += dY[i, j] / (k*k)             # 均分
    return dX

X = rng.standard_normal((4, 4))
for mode in ['max', 'avg']:
    Y = pool2d(X, 2, 2, mode)
    dY = rng.standard_normal(Y.shape)
    dX = pool2d_backward(X, dY, 2, 2, mode)
    # 数值检验：把 dY 作为上游梯度，等价于检验 L = sum(Y*dY)
    L = lambda Xv: np.sum(pool2d(Xv, 2, 2, mode) * dY)
    dX_num = numerical_grad(L, X.copy())
    print(f'{mode} pool 反向 相对误差 = {rel_error(dX, dX_num):.2e}')
    assert rel_error(dX, dX_num) < 1e-6
print('✅ max(只回 argmax) / avg(均分) 池化反向均通过数值检验')

---
# ✏️ 练习区

每题先看题面，在骨架里填 `TODO`，运行**紧跟的自测 cell**（assert 判分）。卡住了看后面的 📖 参考答案。

## ✏️ 练习 1：通用输出尺寸 + same padding

实现 `same_pad(k)` 返回让 `stride=1` 卷积保持尺寸的 padding（假设奇数核），并实现 `conv_out_hw(H, W, kh, kw, s, p)` 返回 `(oh, ow)`。

In [ ]:
def same_pad(k):
    # TODO: 返回奇数核 same 卷积的 padding（一个整数）
    raise NotImplementedError

def conv_out_hw(H, W, kh, kw, s=1, p=0):
    # TODO: 返回 (oh, ow)，用 floor((x+2p-k)/s)+1
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert same_pad(3) == 1 and same_pad(5) == 2 and same_pad(7) == 3
assert conv_out_hw(8, 8, 3, 3, 1, 0) == (6, 6)        # valid
assert conv_out_hw(8, 8, 3, 3, 1, same_pad(3)) == (8, 8)  # same
assert conv_out_hw(32, 32, 5, 5, 2, 2) == (16, 16)    # floor((32+4-5)/2)+1=16
assert conv_out_hw(7, 7, 2, 2, 2, 0) == (3, 3)        # 不整除截断
print('✅ 练习 1 通过：输出尺寸 + same padding 正确')

## ✏️ 练习 2：感受野计算

给定一串卷积/池化层配置 `[(k, s), ...]`，从零算出最终输出一个像素的**理论感受野**。

递推：`jump` 初始 1，`rf` 初始 1；每层 `rf += (k-1)*jump`，然后 `jump *= s`。

In [ ]:
def receptive_field(layers):
    '''layers: [(k, s), ...] 按从输入到输出顺序。返回最终理论感受野(int)。'''
    rf, jump = 1, 1
    # TODO: 遍历每层，按 rf += (k-1)*jump; jump *= s 更新
    raise NotImplementedError
    return rf

In [ ]:
# —— 练习 2 自测 ——
# 三层 3x3 stride1：RF = 1+2+2+2 = 7
assert receptive_field([(3,1),(3,1),(3,1)]) == 7
# 一层 5x5：RF=5；两层 3x3 应等于一个 5x5 的 RF
assert receptive_field([(5,1)]) == 5
assert receptive_field([(3,1),(3,1)]) == 5     # 两个 3x3 == 一个 5x5 的感受野
# 含 stride/池化：conv3x3 s1 -> pool2 s2 -> conv3x3 s1
# rf: 1 ->(3,1):3 ->(2,2):3+(2-1)*1=4, jump=2 ->(3,1):4+(3-1)*2=8
assert receptive_field([(3,1),(2,2),(3,1)]) == 8
print('✅ 练习 2 通过：感受野递推正确（两个 3x3 = 一个 5x5）')

## ✏️ 练习 3：卷积层参数量

实现 `layer_params(C_in, C_out, kh, kw, bias=True)` 返回参数量。

用它对照 **VGG 第一层**（输入 3 通道、输出 64、3×3）的参数量 = 1792。

In [ ]:
def layer_params(C_in, C_out, kh, kw, bias=True):
    # TODO: 权重 C_out*C_in*kh*kw, 有 bias 再加 C_out
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert layer_params(3, 64, 3, 3) == 1792       # VGG conv1_1: 3*64*9+64
assert layer_params(64, 64, 3, 3) == 36928     # VGG conv1_2
assert layer_params(3, 64, 3, 3, bias=False) == 1728
# 1x1 卷积省参数：512->512 的 1x1 vs 3x3
assert layer_params(512, 512, 1, 1) == 262656
assert layer_params(512, 512, 3, 3) == 2359808  # 3x3 是 1x1 的 9 倍多
print('✅ 练习 3 通过：参数量计算正确（对上了真实 VGG 层）')

## ✏️ 练习 4：avg 池化的反向

不看第 7 节代码，自己实现 **avg 池化的反向**：把每个输出梯度均分回窗口内每个输入位置。

In [ ]:
def avgpool_backward(X, dY, k=2, stride=2):
    dX = np.zeros_like(X)
    oh, ow = dY.shape
    # TODO: 对每个 (i,j)，把 dY[i,j]/(k*k) 加到对应输入窗口的每个位置
    raise NotImplementedError
    return dX

In [ ]:
# —— 练习 4 自测 ——
X = rng.standard_normal((4, 4))
Y = pool2d(X, 2, 2, 'avg')
dY = rng.standard_normal(Y.shape)
dX = avgpool_backward(X, dY, 2, 2)
L = lambda Xv: np.sum(pool2d(Xv, 2, 2, 'avg') * dY)
assert rel_error(dX, numerical_grad(L, X.copy())) < 1e-6
# 均分性质：每个输出位置贡献的总梯度 = dY[i,j]（k*k 份各 1/(k*k)）
assert np.allclose(dX.sum(), dY.sum())
print('✅ 练习 4 通过：avg 池化反向均分正确，且梯度总量守恒')

---
# 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def same_pad(k):
    return (k - 1) // 2

def conv_out_hw(H, W, kh, kw, s=1, p=0):
    oh = (H + 2*p - kh) // s + 1
    ow = (W + 2*p - kw) // s + 1
    return (oh, ow)

In [ ]:
# 练习 2
def receptive_field(layers):
    rf, jump = 1, 1
    for k, s in layers:
        rf += (k - 1) * jump
        jump *= s
    return rf

In [ ]:
# 练习 3
def layer_params(C_in, C_out, kh, kw, bias=True):
    n = C_out * C_in * kh * kw
    if bias:
        n += C_out
    return n

In [ ]:
# 练习 4
def avgpool_backward(X, dY, k=2, stride=2):
    dX = np.zeros_like(X)
    oh, ow = dY.shape
    for i in range(oh):
        for j in range(ow):
            r, cc = i*stride, j*stride
            dX[r:r+k, cc:cc+k] += dY[i, j] / (k*k)
    return dX

---
# 🧪 真实数据胶囊：在 optdigits 真实手写数字上做边缘检测

用 `sklearn` 的真实 **8×8 手写数字**，亲手用一个 **Sobel 边缘检测核**做卷积，看卷积如何提取真实图像的结构（笔画边缘）。

这把抽象的「滑窗加权和」接到了**真实数据**：边缘检测正是卷积最初被发明出来做的事。

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
img = digits.images[0]            # 第 0 张, 数字 0, shape (8,8)
print('数字标签 =', digits.target[0], '| 图像尺寸', img.shape)
# Sobel 垂直边缘核（检测水平方向的灰度突变）
sobel_x = np.array([[-1., 0, 1],
                    [-2., 0, 2],
                    [-1., 0, 1]])
edges = corr2d(img, sobel_x)      # 用我们自己的互相关
print('边缘图尺寸', edges.shape, '(8-3+1=6)')
# 对拍 scipy 确认我们在真实数据上也对
from scipy.signal import correlate2d
check_allclose('真实数字上 corr2d vs scipy', edges, correlate2d(img, sobel_x, 'valid'))
print('原图(取整):'); print(img.astype(int))
print('Sobel-x 边缘响应(取整):'); print(edges.astype(int))
print('✅ 在真实手写数字上，卷积成功提取了笔画的垂直边缘')

**🧪 胶囊练习**：实现 `edge_magnitude(img)`：分别用 Sobel-x 和 Sobel-y 卷积，再算每个位置的**梯度幅值** `sqrt(gx^2 + gy^2)`，得到与方向无关的边缘强度图。

In [ ]:
sobel_y = sobel_x.T               # 水平边缘核 = 垂直核的转置

def edge_magnitude(img):
    # TODO: gx = corr2d(img, sobel_x); gy = corr2d(img, sobel_y)
    #       返回 sqrt(gx**2 + gy**2)
    raise NotImplementedError

In [ ]:
# 自测
mag = edge_magnitude(img)
assert mag.shape == (6, 6)
assert np.all(mag >= 0)                 # 幅值非负
gx = corr2d(img, sobel_x); gy = corr2d(img, sobel_y)
assert np.allclose(mag, np.sqrt(gx**2 + gy**2))
print('边缘强度图(取整):'); print(mag.astype(int))
print('✅ 胶囊练习通过：方向无关的边缘强度 = sqrt(gx^2+gy^2)')

In [ ]:
# 📖 胶囊参考答案
def edge_magnitude(img):
    gx = corr2d(img, sobel_x)
    gy = corr2d(img, sobel_y)
    return np.sqrt(gx**2 + gy**2)

### 小结
- **卷积 = 一小块核滑遍全图、逐位相乘求和**；框架的「卷积」其实是**不翻转核的互相关**（核是学的，翻转无所谓）。
- 输出尺寸 `floor((in+2p-k)/s)+1`；参数量 `C_out*C_in*kh*kw+C_out`，**与图像尺寸无关**。
- 感受野随深度增长（两个 3×3 = 一个 5×5）；池化降采样并提供局部平移不变性。
- **反向**：对核 `dK=corr(X,dY)`、对输入 `dX=full_conv(dY, 翻转核)`；max 池化只回 argmax、avg 池化均分——全部用数值梯度验证过。

下一站：**模块 02 · CNN 架构演化** —— 把这些零件组装成 LeNet→AlexNet→VGG→ResNet，并搞懂残差连接为何能堆到上百层。